In [2]:
# First let's do an import. If you get an Import Error, double check that your Kernel is correct..

from dotenv import load_dotenv


In [3]:
# Next it's time to load the API keys into environment variables
# If this returns false, see the next cell!

load_dotenv(override=True)

True

In [4]:
import requests
import os

class OpenRouterChatCompletions:
    def __init__(self, api_key):
        self.api_key = api_key
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "HTTP-Referer": "http://localhost",
            "X-Title": "My Agent App",
            "Content-Type": "application/json"
        }

    def create(self, model, messages, max_tokens=256):
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=self.headers,
            json={
                "model": model,
                "messages": messages,
                "max_tokens": max_tokens
            }
        )

        data = response.json()

        if "choices" not in data:
            print("Full API Response:", data)
            raise Exception("OpenRouter did not return choices.")

        return data


class OpenRouterClient:
    def __init__(self, api_key):
        self.chat = self.Chat(api_key)

    class Chat:
        def __init__(self, api_key):
            self.completions = OpenRouterChatCompletions(api_key)


# Instantiate exactly like OpenAI
openai = OpenRouterClient(os.getenv("OPENROUTER_API_KEY"))


### Wait, did that just output `False`??

If so, the most common reason is that you didn't save your `.env` file after adding the key! Be sure to have saved.

Also, make sure the `.env` file is named precisely `.env` and is in the project root directory (`agents`)

By the way, your `.env` file should have a stop symbol next to it in Cursor on the left, and that's actually a good thing: that's Cursor saying to you, "hey, I realize this is a file filled with secret information, and I'm not going to send it to an external AI to suggest changes, because your keys should not be shown to anyone else."

In [5]:
# Check the key - if you're not using OpenAI, check whichever key you're using! Ollama doesn't need a key.

import os
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set - please head to the troubleshooting guide in the setup folder")
    


OpenRouter API Key exists and begins sk-or-v1


In [6]:
# And now - the all important import statement
# If you get an import error - head over to troubleshooting in the Setup folder
# Even for other LLM providers like Gemini, you still use this OpenAI import - see Guide 9 for why

from openai import OpenAI

In [7]:
import requests
import os

# Load API key
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY environment variable is not set.")

# Standard OpenRouter headers
headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "HTTP-Referer": "http://localhost",   # Replace with your actual URL for production
    "X-Title": "My Agent App"
}

def chat_completion(messages, model="openai/gpt-4.1"):
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": 256   # REQUESTED TOKEN BUDGET (much smaller)
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=30
    )

    if response.status_code != 200:
        print("Status Code:", response.status_code)
        print("Response:", response.text)
        raise Exception("OpenRouter API returned an error")

    return response.json()




# Example usage
if __name__ == "__main__":
    reply = chat_completion([
        {"role": "user", "content": "Hello!"}
    ])

    print(reply["choices"][0]["message"]["content"])


Hello! How can I help you today? 😊


In [8]:
# Create a list of messages in the familiar OpenAI format

messages = [{"role": "user", "content": "What is 2+2?"}]

In [9]:
import requests
import os

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "HTTP-Referer": "http://localhost",
    "X-Title": "My Agent App"
}

def chat_completion(messages, model="openai/gpt-4.1-mini"):
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json={
            "model": model,
            "messages": messages,
            "max_tokens": 256
        }
    )

    data = response.json()

    if "choices" not in data:
        print("Full API Response:", data)
        raise Exception("OpenRouter did not return 'choices'. Check the error message above.")

    return data["choices"][0]["message"]["content"]


# Call it
response = chat_completion(messages)

print(response)


2 + 2 = 4


In [10]:
# And now - let's ask for a question:

question = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."
messages = [{"role": "user", "content": question}]


In [11]:
response = openai.chat.completions.create(
    model="openai/gpt-4.1-mini",   # ✅ Correct model name for OpenRouter
    messages=messages
)

question = response["choices"][0]["message"]["content"]

print(question)

A farmer needs to cross a river with a wolf, a goat, and a cabbage. His boat can only carry him and one item at a time. If left alone, the wolf will eat the goat, and the goat will eat the cabbage. How can the farmer transport all three safely across the river?


In [12]:
# form a new messages list
messages = [{"role": "user", "content": question}]


In [13]:
import requests
import os

class OpenRouterMessage:
    def __init__(self, content):
        self.content = content

class OpenRouterChoice:
    def __init__(self, message):
        self.message = OpenRouterMessage(message)

class OpenRouterResponse:
    def __init__(self, data):
        # Convert OpenRouter JSON into OpenAI-like objects
        self.choices = [
            OpenRouterChoice(choice["message"]["content"])
            for choice in data.get("choices", [])
        ]

class OpenRouterChatCompletions:
    def __init__(self, api_key):
        self.api_key = api_key
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "HTTP-Referer": "http://localhost",
            "X-Title": "My Agent App",
        }

    def create(self, model, messages, max_tokens=256):
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=self.headers,
            json={
                "model": model,
                "messages": messages,
                "max_tokens": max_tokens
            }
        )

        data = response.json()

        if "choices" not in data:
            print("Full API Error:", data)
            raise Exception("OpenRouter error")

        return OpenRouterResponse(data)


class OpenRouterClient:
    def __init__(self, api_key):
        self.chat = self.Chat(api_key)

    class Chat:
        def __init__(self, api_key):
            self.completions = OpenRouterChatCompletions(api_key)


# Instantiate for user code to match OpenAI:
openai = OpenRouterClient(os.getenv("OPENROUTER_API_KEY"))


In [14]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Give me a short question."}
]

response = openai.chat.completions.create(
    model="openai/gpt-4.1-mini",
    messages=messages
)

answer = response.choices[0].message.content
print("Model output:", answer)


Model output: What is your favorite book?


In [15]:
from IPython.display import Markdown, display
display(Markdown(answer))


What is your favorite book?

# Congratulations!

That was a small, simple step in the direction of Agentic AI, with your new environment!

Next time things get more interesting...

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try this commercial application:<br/>
            First ask the LLM to pick a business area that might be worth exploring for an Agentic AI opportunity.<br/>
            Then ask the LLM to present a pain-point in that industry - something challenging that might be ripe for an Agentic solution.<br/>
            Finally have 3 third LLM call propose the Agentic AI solution. <br/>
            We will cover this at up-coming labs, so don't worry if you're unsure.. just give it a try!
            </span>
        </td>
    </tr>
</table>

In [16]:
# 1. Create the initial messages
messages = [
    {"role": "user", "content": "Give me a simple business idea."}
]

# 2. Make the first call
response = openai.chat.completions.create(
    model="openai/gpt-4.1-mini",
    messages=messages
)

# 3. Extract the business idea
business_idea = response.choices[0].message.content
print("Business Idea:", business_idea)

# 4. Use the business idea in the next message
messages.append({"role": "assistant", "content": business_idea})
messages.append({
    "role": "user",
    "content": f"Please explain this business idea in more detail: {business_idea}"
})

# 5. Make the second call
response2 = openai.chat.completions.create(
    model="openai/gpt-4.1-mini",
    messages=messages
)

detailed_explanation = response2.choices[0].message.content
print("\nDetailed Explanation:", detailed_explanation)


Business Idea: Sure! Here’s a simple business idea:

**Mobile Car Wash Service**

- **What:** A convenient car washing service that goes to customers’ homes or workplaces.
- **Why:** Many people don’t have time to visit a car wash, so bringing the service to them offers convenience.
- **How:** Use eco-friendly cleaning supplies and minimal water usage to attract environmentally-conscious customers.
- **Start-up Costs:** Basic cleaning equipment, cleaning products, and a reliable vehicle.

Would you like tips on how to get started with this?

Detailed Explanation: Certainly! Here’s a more detailed explanation of the **Mobile Car Wash Service** business idea:

### 1. What is it?
The Mobile Car Wash Service is a business where you provide car cleaning and detailing directly at customers’ preferred locations, such as their home, office, or apartment complex. Instead of customers having to drive to a fixed car wash location, your team brings all necessary equipment to clean their vehicles o